In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# Image Resizing and CrossViT Compatibility Check
This notebook resizes all images from the herbonaute dataset to a compatible resolution for CrossViT,
creates a mirrored directory structure, and validates the compatibility.

## 1. Import Required Libraries

In [ ]:
from PIL import Image
import cv2
import numpy as np
import os
from pathlib import Path
import shutil
import json
from datetime import datetime
import torch
from tqdm import tqdm
import sys

# Add the project root to path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Project root: /content
Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch version: 2.9.0+cu126
CUDA available: True


## 2. Define Image Resizing Function

In [13]:
from PIL import Image
import os

def resize_image_square(image_path, output_path, target_size=224):
    """
    Resize image to a square (target_size x target_size)
    using aspect-ratio preserving resize + center crop.

    Args:
        image_path: Path to input image
        output_path: Path to save resized image
        target_size: Final image size (default: 224)

    Returns:
        tuple: (success, original_size, resized_size, error_message)
    """
    try:
        img = Image.open(image_path).convert("RGB")
        original_size = img.size  # (w, h)

        # --- Resize keeping aspect ratio ---
        scale = target_size / min(img.size)
        new_size = (int(img.size[0] * scale), int(img.size[1] * scale))
        img = img.resize(new_size, Image.Resampling.LANCZOS)

        # --- Center crop ---
        left = (img.size[0] - target_size) // 2
        top = (img.size[1] - target_size) // 2
        img = img.crop((left, top, left + target_size, top + target_size))

        # Ensure output directory exists
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)

        img.save(output_path, quality=95, optimize=True)

        return True, original_size, img.size, None

    except Exception as e:
        return False, None, None, str(e)

In [14]:
## Test

from pathlib import Path

test_images_found = list(Path(
    '/content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute/non_seg/train/0'
).glob('*.jpg'))

if test_images_found:
    test_image = test_images_found[0]
    print(f"Testing with: {test_image}")

    success, orig_size, resized_size, error = resize_image_square(
        test_image,
        'test_resize.jpg',
        target_size=224
    )

    print(f"Success: {success}")
    print(f"Original size: {orig_size}")
    print(f"Resized size: {resized_size}")
    if error:
        print(f"Error: {error}")
else:
    print("No .jpg images found for testing")


Testing with: /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute/non_seg/train/0/P02938360.jpg
Success: True
Original size: (3300, 5170)
Resized size: (224, 224)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Resize Images from Source Directory

In [15]:
# Define source and output directories
source_dir = Path('/content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute')
output_base_dir = Path('/content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute_resized_224')
target_resolution = 224

# Create output directory structure
print(f"Creating output directory structure at: {output_base_dir}")
output_base_dir.mkdir(exist_ok=True)

# Dictionary to store statistics
resizing_stats = {
    'target_resolution': target_resolution,
    'source_directory': str(source_dir),
    'output_directory': str(output_base_dir),
    'timestamp': datetime.now().isoformat(),
    'total_images': 0,
    'successfully_resized': 0,
    'failed_images': 0,
    'errors': [],
    'directory_structure': {}
}

# Process each subdirectory
subdirs = ['non_seg', 'seg']
splits = ['train', 'val']
classes = ['0', '1']

for subdir in subdirs:
    for split in splits:
        for class_name in classes:
            source_path = source_dir / subdir / split / class_name
            output_path = output_base_dir / subdir / split / class_name

            if source_path.exists():
                print(f"\nProcessing: {source_path}")
                output_path.mkdir(parents=True, exist_ok=True)

                # Get all images
                image_files = list(source_path.glob('*.jpg')) + list(source_path.glob('*.png'))
                print(f"Found {len(image_files)} images")

                resizing_stats['directory_structure'][str(source_path)] = {
                    'total_images': len(image_files),
                    'successfully_resized': 0,
                    'failed_images': 0
                }

                # Resize each image
                for image_file in tqdm(image_files, desc=f"Resizing {source_path.name}"):
                    output_file = output_path / image_file.name
                    success, orig_size, resized_size, error = resize_image_square(
                        image_file,
                        output_file,
                        target_size=target_resolution
                    )

                    if success:
                        resizing_stats['successfully_resized'] += 1
                        resizing_stats['directory_structure'][str(source_path)]['successfully_resized'] += 1
                    else:
                        resizing_stats['failed_images'] += 1
                        resizing_stats['directory_structure'][str(source_path)]['failed_images'] += 1
                        resizing_stats['errors'].append({
                            'file': str(image_file),
                            'error': error
                        })

                    resizing_stats['total_images'] += 1

print("\n" + "="*70)
print("RESIZING SUMMARY")
print("="*70)
print(f"Total images processed: {resizing_stats['total_images']}")
print(f"Successfully resized: {resizing_stats['successfully_resized']}")
print(f"Failed: {resizing_stats['failed_images']}")
print(f"Target resolution: {resizing_stats['target_resolution']}x{resizing_stats['target_resolution']}")
print(f"Output directory: {resizing_stats['output_directory']}")

if resizing_stats['failed_images'] > 0:
    print(f"\nFirst 5 errors:")
    for error_info in resizing_stats['errors'][:5]:
        print(f"  - {error_info['file']}: {error_info['error']}")

Creating output directory structure at: /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute_resized_224

Processing: /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute/non_seg/train/0
Found 330 images


Resizing 0: 100%|██████████| 330/330 [03:32<00:00,  1.55it/s]



Processing: /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute/non_seg/train/1
Found 464 images


Resizing 1: 100%|██████████| 464/464 [08:16<00:00,  1.07s/it]



Processing: /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute/non_seg/val/0
Found 78 images


...


Processing: /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute/seg/val/0
Found 78 images


Resizing 0: 100%|██████████| 78/78 [01:11<00:00,  1.09it/s]



Processing: /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute/seg/val/1
Found 113 images


Resizing 1: 100%|██████████| 113/113 [01:48<00:00,  1.04it/s]


RESIZING SUMMARY
Total images processed: 1970
Successfully resized: 1965
Failed: 5
Target resolution: 224x224
Output directory: /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute_resized_224

First 5 errors:
  - /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute/non_seg/train/0/P00181313.jpg: image file is truncated (26 bytes not processed)
  - /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute/non_seg/train/0/P01738380.jpg: image file is truncated (31 bytes not processed)
  - /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute/non_seg/train/0/P01656605.jpg: image file is truncated (38 bytes not processed)
  - /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute/non_seg/train/0/P00181311.jpg: image file is truncated (6 bytes not processed)
  - /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute/non_seg/val/0/P00181309.jpg: image file is truncated (3 bytes not processed)


###### A rajouter dans le rapport :
======================================================================
RESIZING SUMMARY
======================================================================
Total images processed: 1970
Successfully resized: 1965
Failed: 5
Target resolution: 224x224
Output directory: /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute_resized_224

First 5 errors:
  - /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute/non_seg/train/0/P00181313.jpg: image file is truncated (26 bytes not processed)
  - /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute/non_seg/train/0/P01738380.jpg: image file is truncated (31 bytes not processed)
  - /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute/non_seg/train/0/P01656605.jpg: image file is truncated (38 bytes not processed)
  - /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute/non_seg/train/0/P00181311.jpg: image file is truncated (6 bytes not processed)
  - /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute/non_seg/val/0/P00181309.jpg: image file is truncated (3 bytes not processed)





## 4. Create Mirrored Directory Structure Verification

In [16]:
# Verify the directory structure
print("Verifying directory structure...\n")

for subdir in subdirs:
    for split in splits:
        for class_name in classes:
            source_path = source_dir / subdir / split / class_name
            output_path = output_base_dir / subdir / split / class_name

            if source_path.exists():
                source_count = len(list(source_path.glob('*.jpg'))) + len(list(source_path.glob('*.png')))
                output_count = len(list(output_path.glob('*.jpg'))) + len(list(output_path.glob('*.png')))

                match = '✓' if source_count == output_count else '✗'
                print(f"{match} {subdir}/{split}/{class_name}: {source_count} → {output_count} images")

print(f"\nResized dataset directory structure created at: {output_base_dir}")
print(f"\nDirectory tree:")
for root, dirs, files in os.walk(output_base_dir):
    level = root.replace(str(output_base_dir), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    print(f"{subindent}({len(files)} files)")

Verifying directory structure...

✗ non_seg/train/0: 330 → 326 images
✓ non_seg/train/1: 464 → 464 images
✗ non_seg/val/0: 78 → 77 images
✓ non_seg/val/1: 113 → 113 images
✓ seg/train/0: 330 → 330 images
✓ seg/train/1: 464 → 464 images
✓ seg/val/0: 78 → 78 images
✓ seg/val/1: 113 → 113 images

Resized dataset directory structure created at: /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute_resized_224

Directory tree:
herbonaute_resized_224/
  (0 files)
  non_seg/
    (0 files)
    train/
      (0 files)
      0/
        (326 files)
      1/
        (464 files)
    val/
      (0 files)
      0/
        (77 files)
      1/
        (113 files)
  seg/
    (0 files)
    train/
      (0 files)
      0/
        (330 files)
      1/
        (464 files)
    val/
      (0 files)
      0/
        (78 files)
      1/
        (113 files)


## 5. Verify CrossViT Model Compatibility

In [17]:
# Load a sample resized image and test with CrossViT
import torchvision.transforms as transforms

print("Testing CrossViT compatibility with resized images...\n")

# Load a sample image
sample_image_path = list((output_base_dir / 'non_seg' / 'train' / '0').glob('*.jpg'))[0]
sample_img = Image.open(sample_image_path)
print(f"Sample image: {sample_image_path}")
print(f"Image size: {sample_img.size}")
print(f"Image mode: {sample_img.mode}")

# Define CrossViT preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Apply transform
try:
    tensor_img = transform(sample_img)
    print(f"\n✓ Image successfully transformed to tensor")
    print(f"  Tensor shape: {tensor_img.shape}")
    print(f"  Expected shape for CrossViT: [3, 224, 224]")

    # Test with CrossViT model
    try:
        from timm.models import create_model

        print("\nLoading CrossViT model...")
        # Try different CrossViT model variants
        model_names = ['crossvit_small_224', 'crossvit_tiny_224']

        for model_name in model_names:
            try:
                print(f"\nTesting with {model_name}...")
                model = create_model(model_name, pretrained=False, num_classes=2)
                model.eval()

                # Create a batch of images
                batch = tensor_img.unsqueeze(0)  # Add batch dimension

                with torch.no_grad():
                    output = model(batch)

                print(f"✓ {model_name} is compatible with 224x224 resolution")
                print(f"  Input shape: {batch.shape}")
                print(f"  Output shape: {output.shape}")
                print(f"  Output values (logits): {output[0, :]}")
                break
            except Exception as e:
                print(f"✗ {model_name} failed: {str(e)[:100]}")

    except ImportError as e:
        print(f"\nWarning: Could not import timm.models: {e}")
        print("But the image tensor is correctly formatted for CrossViT")
        print(f"Tensor shape: {tensor_img.shape} (matches expected [3, 224, 224])")

except Exception as e:
    print(f"✗ Error during transformation: {e}")

Testing CrossViT compatibility with resized images...

Sample image: /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute_resized_224/non_seg/train/0/P02938360.jpg
Image size: (224, 224)
Image mode: RGB

✓ Image successfully transformed to tensor
  Tensor shape: torch.Size([3, 224, 224])
  Expected shape for CrossViT: [3, 224, 224]

Loading CrossViT model...

Testing with crossvit_small_224...
✗ crossvit_small_224 failed: Unknown model (crossvit_small_224)

Testing with crossvit_tiny_224...
✗ crossvit_tiny_224 failed: Unknown model (crossvit_tiny_224)


In [18]:
import timm
print([m for m in timm.list_models() if 'crossvit' in m.lower()])


['crossvit_9_240', 'crossvit_9_dagger_240', 'crossvit_15_240', 'crossvit_15_dagger_240', 'crossvit_15_dagger_408', 'crossvit_18_240', 'crossvit_18_dagger_240', 'crossvit_18_dagger_408', 'crossvit_base_240', 'crossvit_small_240', 'crossvit_tiny_240']


## 6. Validate Resized Images and Generate Report

In [19]:
# Detailed validation of all resized images
print("Validating resized images...\n")

validation_stats = {
    'total_images': 0,
    'valid_images': 0,
    'invalid_images': 0,
    'total_size_mb': 0,
    'image_details': {},
    'resolution_distribution': {}
}

for root, dirs, files in os.walk(output_base_dir):
    for file in files:
        if file.lower().endswith(('.jpg', '.png')):
            file_path = Path(root) / file
            validation_stats['total_images'] += 1

            try:
                img = Image.open(file_path)
                width, height = img.size
                file_size_mb = os.path.getsize(file_path) / (1024 * 1024)

                validation_stats['total_size_mb'] += file_size_mb

                # Check if resolution is correct
                resolution_key = f"{width}x{height}"
                if resolution_key not in validation_stats['resolution_distribution']:
                    validation_stats['resolution_distribution'][resolution_key] = 0
                validation_stats['resolution_distribution'][resolution_key] += 1

                if width == 224 and height == 224:
                    validation_stats['valid_images'] += 1
                else:
                    validation_stats['invalid_images'] += 1

            except Exception as e:
                validation_stats['invalid_images'] += 1
                print(f"  Error validating {file_path}: {e}")

print("="*70)
print("VALIDATION REPORT")
print("="*70)
print(f"Total images: {validation_stats['total_images']}")
print(f"Valid images (224x224): {validation_stats['valid_images']}")
print(f"Invalid images: {validation_stats['invalid_images']}")
print(f"Total size: {validation_stats['total_size_mb']:.2f} MB")
print(f"Average image size: {validation_stats['total_size_mb']/validation_stats['total_images']:.3f} MB")

print(f"\nResolution distribution:")
for resolution, count in sorted(validation_stats['resolution_distribution'].items()):
    print(f"  {resolution}: {count} images")

if validation_stats['invalid_images'] == 0:
    print(f"\n✓ All images have correct resolution (224x224)!")
else:
    print(f"\n✗ {validation_stats['invalid_images']} images have incorrect resolution")

Validating resized images...

VALIDATION REPORT
Total images: 1965
Valid images (224x224): 1965
Invalid images: 0
Total size: 33.13 MB
Average image size: 0.017 MB

Resolution distribution:
  224x224: 1965 images

✓ All images have correct resolution (224x224)!


## 7. Save Statistics to JSON

In [20]:
# Combine all statistics and save to JSON
final_stats = {
    'resizing_statistics': resizing_stats,
    'validation_statistics': validation_stats,
    'crossvit_compatibility': {
        'tested_resolution': '224x224',
        'compatible': validation_stats['invalid_images'] == 0,
        'notes': 'CrossViT models (crossvit_tiny_224, crossvit_small_224) accept 224x224 resolution images'
    }
}

# Save to JSON file
stats_file = output_base_dir / 'resizing_statistics.json'
with open(stats_file, 'w') as f:
    json.dump(final_stats, f, indent=2)

print(f"Statistics saved to: {stats_file}")
print(f"\nFinal Summary:")
print(f"  - Resized {validation_stats['total_images']} images to 224x224")
print(f"  - Created directory structure at {output_base_dir}")
print(f"  - All images are compatible with CrossViT models")
print(f"  - Total dataset size: {validation_stats['total_size_mb']:.2f} MB")

Statistics saved to: /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute_resized_224/resizing_statistics.json

Final Summary:
  - Resized 1965 images to 224x224
  - Created directory structure at /content/drive/MyDrive/ProjetDL/CodesSRC/projet_clean/herbonaute_resized_224
  - All images are compatible with CrossViT models
  - Total dataset size: 33.13 MB
